In [18]:
#code snippet 1
import pandas as pd
import numpy as np
# Specify the path to your CSV file
url = 'https://docs.google.com/spreadsheets/d/1XSBE7VwTJ-xB300r2JAwZesaAbGTLvMgS4qRxXfDjys/pub?gid=0&single=true&output=csv'
# Read the CSV file into a pandas DataFrame
data = pd.read_csv(url)

# Display the first few rows of the DataFrame to verify
display(data.head())

,Customer ID,Title,First Name,Middle Name,Last Name,Suffix,Street Address1,Street Address2,City-ZipCode-State,Birth Date,Education Level,Occupation,Gender,Marital Status,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Avg Monthly Spend,eBook Subscriber Flag
0,11000,NaN,Jon,V,Yang,NaN,3761 N. 14th St,NaN,Cleveland-44101-Ohio,4/8/66,Bachelors,Professional,M,M,1,0,0,2,137947,89,0
1,11001,NaN,Eugene,L,Huang,NaN,2243 W St.,NaN,Seattle-98101-Washington,5/14/65,Bachelors,Professional,M,S,0,1,3,3,101141,117,1
2,11002,NaN,Ruben,NaN,Torres,NaN,5844 Linden Land,NaN,Omaha-68101-Nebraska,8/12/65,Bachelors,Professional,M,M,1,1,3,3,91945,123,0
3,11003,NaN,Christy,NaN,Zhu,NaN,1825 Village Pl.,NaN,Fort Worth-76101-Texas,2/15/68,Bachelors,Professional,F,S,0,1,0,0,86688,50,0
4,11004,NaN,Elizabeth,NaN,Johnson,NaN,7553 Harness Circle,NaN,Oakland-94601-California,8/8/68,Bachelors,Professional,F,S,1,4,5,5,92771,95,1


In [19]:
#code snippet 3
#import missing no package
import missingno as msno

In [20]:
#code snippet 7
#displaying missing data proportion nusing missingno package
msno.bar(data)

<Axes: title={'center': 'Average Age by eBook Subscriber Flag (Top 20)'}>

In [21]:
#code snippet 2.1
# Getting column names
listc=data.columns.tolist()
print(list)
# Sum of missing values per column
missing_sum = data.isnull().sum()
print("Sum of missing values per column:")
print(missing_sum)

# Percentage of missing values per column
missing_percentage = (data.isnull().sum() / len(data)) * 100
print("\nPercentage of missing values per column:")
print(missing_percentage)


<class 'list'>
Sum of missing values per column:
Customer ID                       0
Title                         16431
First Name                        0
Middle Name                    6985
Last Name                         0
Suffix                        16517
Street Address1                   0
Street Address2               16243
City-ZipCode-State                0
Birth Date                        0
Education Level                   0
Occupation                        0
Gender                            0
Marital Status                    0
Home Owner Status                 0
Number of Cars Owned              0
Number of Children At Home        0
Total Number of Children          0
Annual Income                     0
Avg Monthly Spend                 0
eBook Subscriber Flag             0
dtype: int64

Percentage of missing values per column:
Customer ID                    0.000000
Title                         99.467280
First Name                     0.000000
Middle Name         

In [22]:
#code snippet 3
#missimng more than 20 % data
#drop Title, Middle Name, Suffix, Street Address2
data80=data.drop(['Title', 'Middle Name', 'Suffix', 'Street Address2'], axis=1)
data80.dtypes

,0
Customer ID,int64
First Name,object
Last Name,object
Street Address1,object
City-ZipCode-State,object
Birth Date,object
Education Level,object
Occupation,object
Gender,object
Marital Status,object


In [23]:
#code snippet 4
#qualitative data preparation
#remove noise varaibles:
dataq=data80.drop(['Customer ID', 'First Name', 'Last Name', 'Street Address1'], axis=1)
#separate City-ZipCode-State into 3 varaibles:
dataq[['City', 'ZipCode', 'State']]=dataq['City-ZipCode-State'].str.split('-', expand=True)
#drop City-ZipCode-State, remove noise variable zip
dataq=dataq.drop(['City-ZipCode-State'], axis=1)
dataq

,Birth Date,Education Level,Occupation,Gender,Marital Status,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Avg Monthly Spend,eBook Subscriber Flag,City,ZipCode,State
0,4/8/66,Bachelors,Professional,M,M,1,0,0,2,137947,89,0,Cleveland,44101,Ohio
1,5/14/65,Bachelors,Professional,M,S,0,1,3,3,101141,117,1,Seattle,98101,Washington
2,8/12/65,Bachelors,Professional,M,M,1,1,3,3,91945,123,0,Omaha,68101,Nebraska
3,2/15/68,Bachelors,Professional,F,S,0,1,0,0,86688,50,0,Fort Worth,76101,Texas
4,8/8/68,Bachelors,Professional,F,S,1,4,5,5,92771,95,1,Oakland,94601,California
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16514,3/22/65,Bachelors,Professional,F,M,1,4,5,5,101542,101,0,San Antonio,78201,Texas
16515,4/2/36,Partial College,Professional,F,S,1,2,0,3,46549,46,0,Pittsburgh,15201,Pennsylvania
16516,1/1/40,Bachelors,Management,M,M,1,2,0,5,133053,79,0,Honolulu,96801,Hawaii
16517,10/20/46,High School,Skilled Manual,M,M,1,2,0,4,31930,65,0,Anaheim,92801,California


In [24]:
#Code snippet 5
#Feature Engineering
#derive age from Birth Date
from datetime import datetime
import numpy as np
import pandas as pd # Ensure pandas is imported for pd.NA

today = datetime.now()
# Convert 'Birth Date' to datetime objects with explicit format
dataq['Birth Date'] = pd.to_datetime(dataq['Birth Date'], format='%m/%d/%y')

# Correct birth years that are in the future due to two-digit year interpretation
# If a birth year is after the current year, subtract 100 years
dataq.loc[dataq['Birth Date'].dt.year > today.year, 'Birth Date'] -= pd.offsets.DateOffset(years=100)

# Calculate age based on current date, using only the year part
dataq['Age'] = today.year - dataq['Birth Date'].dt.year

# Filter dataq where 'Birth Date' is less than or equal to today
dataq_filtered = dataq[dataq['Birth Date'] <= today]

#drop 'Birth Date' remove redundant variable
predictor=dataq_filtered.drop(['Birth Date'], axis=1)

# Ensure 'eBook Subscriber Flag' is numeric before comparison
predictor['eBook Subscriber Flag'] = predictor['eBook Subscriber Flag'].astype(int)

#derive esubscribey from eBook Subscriber Flag ==1, otherwise NaN
predictor['esubscribey'] = np.where(predictor['eBook Subscriber Flag'] == 1, 1, pd.NA)

# Ensure 'eBook Subscriber Flag' is character after comparison
predictor['eBook Subscriber Flag'] = predictor['eBook Subscriber Flag'].astype(str)

# Display the first few rows with the new 'Age' and 'esubscribey' columns
display(predictor.head())
predictor.dtypes

,Education Level,Occupation,Gender,Marital Status,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Avg Monthly Spend,eBook Subscriber Flag,City,ZipCode,State,Age,esubscribey
0,Bachelors,Professional,M,M,1,0,0,2,137947,89,0,Cleveland,44101,Ohio,59,<NA>
1,Bachelors,Professional,M,S,0,1,3,3,101141,117,1,Seattle,98101,Washington,60,1
2,Bachelors,Professional,M,M,1,1,3,3,91945,123,0,Omaha,68101,Nebraska,60,<NA>
3,Bachelors,Professional,F,S,0,1,0,0,86688,50,0,Fort Worth,76101,Texas,57,<NA>
4,Bachelors,Professional,F,S,1,4,5,5,92771,95,1,Oakland,94601,California,57,1


,0
Education Level,object
Occupation,object
Gender,object
Marital Status,object
Home Owner Status,int64
Number of Cars Owned,int64
Number of Children At Home,int64
Total Number of Children,int64
Annual Income,int64
Avg Monthly Spend,int64


Checking for any remaining future birth dates in the `dataq` DataFrame after conversion and correction steps:

In [25]:
# Code snippet 5.5

from datetime import datetime

today_year = datetime.now()

# Filter for records where the Birth Date year is still in the future
future_birthdates_remaining = dataq[dataq['Birth Date'] > today]

# Display the results
if not future_birthdates_remaining.empty:
    print(f"Found {future_birthdates_remaining.shape[0]} record(s) with future birth years:")
    display(future_birthdates_remaining)
    # Output these records to a new CSV file
    future_birthdates_remaining.to_csv('future_birthdates.csv', index=False)
    print("These records have been saved to 'future_birthdates.csv'.")
else:
    print("No records found with future birth years after correction.")

Found 1 record(s) with future birth years:


,Birth Date,Education Level,Occupation,Gender,Marital Status,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Avg Monthly Spend,eBook Subscriber Flag,City,ZipCode,State,Age
13383,2025-11-23,Bachelors,Professional,F,S,1,0,0,5,118900,38,0,Saint Paul,55101,Minnesota,0


These records have been saved to 'future_birthdates.csv'.


In [26]:
#code snippet 6
#output dataset for Tableau visualization
predictor.to_csv('predictor.csv', index=False)

In [27]:
#code  snippet 7
#installing ydata profiling package for EDA EDA - Exploratory Data Analysis/data profiling
!pip install -U ydata-profiling

In [28]:
#code snippet 8
#import profilereport from ydata_profiling
from ydata_profiling import ProfileReport

In [29]:
#code snippet 9
#generate report
profilereport=ProfileReport(predictor)
profilereport.to_file('profile_report.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 16/16 [00:01<00:00, 12.95it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [30]:
#code snippet 10
!pip install autoviz

In [31]:
#code snippet 11
#from Autoviz.Autoviz_class import Autoviz_class
# Importing Autoviz components
from autoviz.AutoViz_Class import AutoViz_Class

In [32]:
# Code snippet 12
# Starting/instantiating/configuring the Autoviz class
AV = AutoViz_Class()

In [33]:
#code snippet 13
#generating visuals using autoviz_class
AV.AutoViz(predictor, depVar='Avg Monthly Spend', verbose= 2, max_cols_analyzed=31)


Shape of your Data Set loaded: (16518, 16)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
  Printing up to 30 columns (max) in each category:
    Numeric Columns : []
    Integer-Categorical Columns: ['Number of Cars Owned', 'Number of Children At Home', 'Total Number of Children', 'Annual Income', 'Age']
    String-Categorical Columns: ['Education Level', 'Occupation', 'State', 'City', 'ZipCode']
    Factor-Categorical Columns: []
    String-Boolean Columns: ['Gender', 'Marital Status', 'eBook Subscriber Flag']
    Numeric-Boolean Columns: ['Home Owner Status']
    Discrete String Columns: []
    NLP text Columns: []
    Date Time Columns: []
    ID Columns: []
    Columns that will not be considered in modeling: ['esubscribey']
    15

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
Education Level,object,0.000000,0,,,No issue
Occupation,object,0.000000,0,,,No issue
Gender,object,0.000000,0,,,No issue
Marital Status,object,0.000000,0,,,No issue
Home Owner Status,int64,0.000000,0,0.000000,1.000000,No issue
Number of Cars Owned,int64,0.000000,0,0.000000,4.000000,Column has 1137 outliers greater than upper bound (3.50) or lower than lower bound(-0.50). Cap them or remove them.
Number of Children At Home,int64,0.000000,0,0.000000,5.000000,No issue
Total Number of Children,int64,0.000000,0,0.000000,5.000000,No issue
Annual Income,int64,0.000000,93,9482.000000,196511.000000,Column has 7 outliers greater than upper bound (191262.50) or lower than lower bound(-38267.50). Cap them or remove them.
eBook Subscriber Flag,object,0.000000,0,,,No issue


Number of All Scatter Plots = 15
All Plots are saved in ./AutoViz_Plots/Avg Monthly Spend
Time to run AutoViz = 31 seconds 


,Education Level,Occupation,Gender,Marital Status,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,eBook Subscriber Flag,City,ZipCode,State,Age,Avg Monthly Spend
0,Bachelors,Professional,M,M,1,0,0,2,137947,0,Cleveland,44101,Ohio,59,89
1,Bachelors,Professional,M,S,0,1,3,3,101141,1,Seattle,98101,Washington,60,117
2,Bachelors,Professional,M,M,1,1,3,3,91945,0,Omaha,68101,Nebraska,60,123
3,Bachelors,Professional,F,S,0,1,0,0,86688,0,Fort Worth,76101,Texas,57,50
4,Bachelors,Professional,F,S,1,4,5,5,92771,1,Oakland,94601,California,57,95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16514,Bachelors,Professional,F,M,1,4,5,5,101542,0,San Antonio,78201,Texas,60,101
16515,Partial College,Professional,F,S,1,2,0,3,46549,0,Pittsburgh,15201,Pennsylvania,89,46
16516,Bachelors,Management,M,M,1,2,0,5,133053,0,Honolulu,96801,Hawaii,85,79
16517,High School,Skilled Manual,M,M,1,2,0,4,31930,0,Anaheim,92801,California,79,65


In [34]:
#code snippet 14
#generating visuals using autoviz_class for depvar=R_c
AV.AutoViz(dataq, depVar='eBook Subscriber Flag', verbose= 2, max_cols_analyzed=31)

Shape of your Data Set loaded: (16519, 16)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
  Printing up to 30 columns (max) in each category:
    Numeric Columns : []
    Integer-Categorical Columns: ['Number of Cars Owned', 'Number of Children At Home', 'Total Number of Children', 'Annual Income', 'Avg Monthly Spend', 'Age']
    String-Categorical Columns: ['Education Level', 'Occupation', 'State', 'City', 'ZipCode']
    Factor-Categorical Columns: []
    String-Boolean Columns: ['Gender', 'Marital Status']
    Numeric-Boolean Columns: ['Home Owner Status']
    Discrete String Columns: []
    NLP text Columns: []
    Date Time Columns: ['Birth Date']
    ID Columns: []
    Columns that will not be considered in modeling: []
    15 Pred

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
Birth Date,datetime64[ns],0.000000,47,,,Possible date-time colum: transform before modeling step.
Education Level,object,0.000000,0,,,No issue
Occupation,object,0.000000,0,,,No issue
Gender,object,0.000000,0,,,No issue
Marital Status,object,0.000000,0,,,No issue
Home Owner Status,int64,0.000000,0,0.000000,1.000000,No issue
Number of Cars Owned,int64,0.000000,0,0.000000,4.000000,Column has 1137 outliers greater than upper bound (3.50) or lower than lower bound(-0.50). Cap them or remove them.
Number of Children At Home,int64,0.000000,0,0.000000,5.000000,No issue
Total Number of Children,int64,0.000000,0,0.000000,5.000000,No issue
Annual Income,int64,0.000000,93,9482.000000,196511.000000,Column has 7 outliers greater than upper bound (191275.00) or lower than lower bound(-38273.00). Cap them or remove them.


Total Number of Scatter Plots = 21
All Plots are saved in ./AutoViz_Plots/eBook Subscriber Flag
Time to run AutoViz = 40 seconds 


,Birth Date,Education Level,Occupation,Gender,Marital Status,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Avg Monthly Spend,City,ZipCode,State,Age,eBook Subscriber Flag
0,1966-04-08,Bachelors,Professional,M,M,1,0,0,2,137947,89,Cleveland,44101,Ohio,59,0
1,1965-05-14,Bachelors,Professional,M,S,0,1,3,3,101141,117,Seattle,98101,Washington,60,1
2,1965-08-12,Bachelors,Professional,M,M,1,1,3,3,91945,123,Omaha,68101,Nebraska,60,0
3,1968-02-15,Bachelors,Professional,F,S,0,1,0,0,86688,50,Fort Worth,76101,Texas,57,0
4,1968-08-08,Bachelors,Professional,F,S,1,4,5,5,92771,95,Oakland,94601,California,57,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16514,1965-03-22,Bachelors,Professional,F,M,1,4,5,5,101542,101,San Antonio,78201,Texas,60,0
16515,1936-04-02,Partial College,Professional,F,S,1,2,0,3,46549,46,Pittsburgh,15201,Pennsylvania,89,0
16516,1940-01-01,Bachelors,Management,M,M,1,2,0,5,133053,79,Honolulu,96801,Hawaii,85,0
16517,1946-10-20,High School,Skilled Manual,M,M,1,2,0,4,31930,65,Anaheim,92801,California,79,0
